# 04 — Survival Analysis (Cox Proportional Hazards)

Estimates product survival probability given market conditions.
Model: $h(t|X) = h_0(t) \exp(\sum \beta_i X_i)$

**EN** — A Cox model relates market covariates to the *hazard* (instantaneous risk) that the traditional PBX market "dies". We report hazard ratios, survival curves and scenario survival probabilities.

**繁中** — Cox 比例風險模型將市場條件（寬頻、人均 GDP、都市化、PSTN 退場政策）連結到傳統 PBX 市場「消亡」的瞬時風險。本筆記本輸出風險比 (hazard ratio)、存活曲線與情境存活機率。

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.models.survival import (
    fit_cox_model, plot_survival_curves, 
    predict_survival_probability, rank_markets,
    plot_hazard_ratios
)
from src.data.preprocessor import build_product_lifetime_table

In [ ]:
panel = pd.read_csv('data/processed/panel_data.csv')
print(f"Panel loaded: {panel.shape}")
print(f"Columns: {panel.columns.tolist()}")

## 4.1 Build Product Lifetime Table

Product: "Traditional PBX Gateway" — introduced in 2005.

In [ ]:
product_intro_year = 2005
lifetime = build_product_lifetime_table(
    panel,
    product_intro_year=product_intro_year,
    penetration_col='fixed_subs_value',
    death_threshold=0.05,
)
print(f"Lifetime table: {lifetime.shape[0]} countries")
lifetime

**EN — How to read this table.** `market_lifetime` = years from product introduction (2005) until penetration falls below 5% of each country's historical peak. `product_dead=1` means the threshold was actually crossed within the data window; `product_dead=0` is *right-censored* (still alive at the last observed year). Covariates are taken at the introduction year.

**繁中 — 表格判讀。** `market_lifetime` 為自產品導入年 (2005) 起，到市場滲透率跌破各國歷史高峰 5% 為止的年數。`product_dead=1` 表示在資料期間內確實跨越門檻；`product_dead=0` 為*右設限*（在最後觀測年仍存活）。共變量取自導入年的數值。

## 4.2 Fit Cox Proportional Hazards Model

In [ ]:
covariates = ['broadband_value', 'gdp_per_capita_value', 
              'urban_pop_value', 'has_pstn_phaseout']
available_covs = [c for c in covariates if c in lifetime.columns]
print(f"Using covariates: {available_covs}")

model = fit_cox_model(
    lifetime,
    duration_col='market_lifetime',
    event_col='product_dead',
    covariates=available_covs,
    penalizer=0.1,
)
print("CoxPH model fitted successfully.")
print(f"N = {lifetime.shape[0]} countries, events (deaths) = {int(lifetime['product_dead'].sum())}")
model.print_summary()

**EN — Caveat on sample size.** This model is fitted on a small panel (~13 countries). With 4 covariates and a ridge `penalizer=0.1`, coefficient estimates are stabilised but confidence intervals are wide and p-values should be read as *directional*, not confirmatory. Interpret hazard ratios as relative tendencies, not precise effects.

**繁中 — 樣本數警語。** 本模型僅以約 13 個國家的小樣本配適。使用 4 個共變量並加上 ridge 懲罰項 (`penalizer=0.1`) 雖能穩定係數，但信賴區間偏寬、p 值僅供*方向性*參考，不應視為確證。風險比應理解為相對傾向，而非精確效應量。

## 4.3 Hazard Ratios

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_hazard_ratios(model, ax=ax)
plt.tight_layout()
plt.savefig('data/processed/hazard_ratios.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nInterpretation:")
print("  HR > 1 = higher risk (product dies sooner)")
print("  HR < 1 = lower risk (product survives longer)")

**EN.** A hazard ratio (HR) of 1 means no effect. HR > 1 → that covariate accelerates market death (shorter survival); HR < 1 → it prolongs survival. The horizontal bars are 95% confidence intervals on the HR scale; bars crossing the dashed line at 1.0 are not statistically distinguishable from "no effect".

**繁中.** 風險比 (HR) 等於 1 表示無影響。HR > 1 → 該變量加速市場消亡（存活較短）；HR < 1 → 延長存活。水平線段為 HR 尺度上的 95% 信賴區間；跨越 1.0 虛線者，與「無影響」在統計上無法區分。

## 4.4 Survival Curves by Country

In [ ]:
countries = lifetime['country'].tolist()
fig, ax = plt.subplots(figsize=(10, 6))
plot_survival_curves(model, countries, lifetime, t_max=20, ax=ax)
plt.tight_layout()
plt.savefig('data/processed/survival_curves.png', dpi=150, bbox_inches='tight')
plt.show()

**EN.** Each curve $S(t)$ is the model-estimated probability that the PBX market in that country is still "alive" $t$ years after introduction, given its covariates. Curves that fall faster indicate markets expected to sunset sooner. Beyond the largest observed lifetime the curve is *extrapolated* (held flat) — treat the far tail with caution.

**繁中.** 每條曲線 $S(t)$ 為模型估計：在給定共變量下，該國 PBX 市場於導入後第 $t$ 年仍「存活」的機率。下降越快代表預期越早退場。超過最大觀測壽命的部分屬於*外推*（維持水平），尾端需謹慎解讀。

## 4.5 Market Rankings

In [ ]:
for t in [3, 5, 10]:
    rankings = rank_markets(model, lifetime, t=t)
    print(f"\n=== Market Rankings: {t}-year Survival Probability ===")
    print(rankings.to_string(index=False))

**EN.** Countries are ranked by $S(t)$: higher = the traditional PBX market is expected to persist longer (more time to monetise/maintain). These rankings now use the **same decline-based definition of market death** as notebook 03, so the two notebooks are consistent.

**繁中.** 依 $S(t)$ 排名：數值越高代表傳統 PBX 市場預期持續越久（可維運/變現的時間越長）。此排名採用與筆記本 03 **相同的、以衰退期定義的市場消亡**標準，故兩份筆記本結論一致。

## 4.6 Scenario: Product Launch Decision

In [ ]:
# 4.6 Scenario: Product Launch Decision
# Includes Japan and South Korea alongside the original markets.
scenarios = [
    {"name": "Taiwan (no PSTN phaseout)", "covs": {
        'broadband_value': 35.0, 'gdp_per_capita_value': 35000,
        'urban_pop_value': 80.0, 'has_pstn_phaseout': 0}},
    {"name": "Germany (with phaseout)", "covs": {
        'broadband_value': 42.0, 'gdp_per_capita_value': 48000,
        'urban_pop_value': 77.0, 'has_pstn_phaseout': 1}},
    {"name": "India (growing market)", "covs": {
        'broadband_value': 8.0, 'gdp_per_capita_value': 2500,
        'urban_pop_value': 35.0, 'has_pstn_phaseout': 0}},
    {"name": "Japan (historical baseline)", "covs": {
        'broadband_value': 44.0, 'gdp_per_capita_value': 33800,
        'urban_pop_value': 91.8, 'has_pstn_phaseout': 1}},
    {"name": "South Korea (historical baseline)", "covs": {
        'broadband_value': 46.0, 'gdp_per_capita_value': 32400,
        'urban_pop_value': 81.4, 'has_pstn_phaseout': 0}},
]

print("=== Scenario: Product Launch Decision ===")
for sc in scenarios:
    # Keep only covariates used by the fitted model.
    filtered_covs = {k: v for k, v in sc['covs'].items() if k in available_covs}
    s3 = predict_survival_probability(model, filtered_covs, t=3)
    s5 = predict_survival_probability(model, filtered_covs, t=5)
    s10 = predict_survival_probability(model, filtered_covs, t=10)
    print(f"\n{sc['name']}")
    print(f"  3-year survival: {s3:.1%}")
    print(f"  5-year survival: {s5:.1%}")
    print(f"  10-year survival: {s10:.1%}")

**EN — Scenario reading.** Each row is a hypothetical launch market described by its covariates; the model returns the probability the traditional PBX market survives 3/5/10 years. **Japan** (high broadband + urbanisation + an announced PSTN phaseout) and **South Korea** (very high broadband) are added as East-Asian reference points: both are advanced markets where strong broadband substitution implies *lower* long-run survival for legacy PBX, i.e. a shorter monetisation window than a still-growing market like India.

**繁中 — 情境判讀。** 每一列為以共變量描述的假設性上市市場；模型輸出傳統 PBX 市場存活 3/5/10 年的機率。新增**日本**（高寬頻、高都市化、已宣布 PSTN 退場）與**南韓**（極高寬頻）作為東亞參考點：兩者皆為先進市場，強烈的寬頻替代效應意味著傳統 PBX 的長期存活率*較低*，可變現的時間窗短於仍在成長的市場（如印度）。*註：情境共變量為示意性的歷史基準值，僅供相對比較。*